# 03 — Tool Framework：工具抽象与注册表

前两章解决了「怎么和 LLM 说话」的问题。

这章解决另一个核心问题：**Agent 怎么「做事」**。

光靠文字来回，AI 只能说不能动。Agent 之所以能读文件、写代码、搜索网页，是因为它可以**调用工具**——工具是 Agent 操作世界的手。

---

这章不写具体工具（那是第 04/05 章），而是造**工具运行的基础设施**：
- 所有工具遵守的统一接口（抽象基类）
- 工具调用结果的统一格式（ToolResult）
- 工具的注册表（ToolRegistry）——Agent 通过它调用任何工具

---

## 学完这章你能做到

- 理解 LLM 的 function calling 机制：模型怎么「决定」调用工具
- 写一个符合接口的工具，注册进去就能用
- 理解为什么工具要统一返回格式，而不是各自为政

## 1. LLM 的 Function Calling 机制

先搞清楚工具调用的完整流程，再看代码。

```
你：「帮我读一下 main.py 的内容」
          │
          ▼
    发送给 LLM：
    ┌─────────────────────────────────────────┐
    │ messages: [{user: "帮我读一下 main.py"}] │
    │ tools: [{                               │
    │   name: "read_file",                    │  ← 告诉 LLM 有这个工具
    │   description: "读取文件内容",           │
    │   parameters: {path: string, ...}       │
    │ }]                                      │
    └─────────────────────────────────────────┘
          │
          ▼
    LLM 回复：(不是文字，而是工具调用意图)
    ┌─────────────────────────────────────────┐
    │ tool_calls: [{                          │
    │   id: "call_abc123",                    │  ← 调用 ID，后面要对上
    │   name: "read_file",                    │
    │   arguments: {"path": "main.py"}        │
    │ }]                                      │
    └─────────────────────────────────────────┘
          │
          ▼
    你的代码执行工具，把结果发回：
    ┌─────────────────────────────────────────┐
    │ {role: "tool",                          │
    │  tool_call_id: "call_abc123",           │  ← 对应上面的 ID
    │  content: "def main(): ..."             │  ← 文件内容
    │ }                                       │
    └─────────────────────────────────────────┘
          │
          ▼
    LLM 再次回复：(这次是文字)
    「main.py 的内容是：...」
```

**关键点**：
1. 你把工具列表（tool schemas）发给 LLM，LLM 自己决定调没调用、调哪个、传什么参数
2. LLM 不直接执行工具——它只是「说」想调用，**你的代码**负责真正执行
3. 执行完把结果塞回 messages，LLM 看到结果后继续回答

**Tool schema** 就是告诉 LLM「这个工具叫什么、能做什么、需要什么参数」的 JSON 描述。这是工具框架最核心的部分。

## 2. Tool Schema 的格式

OpenAI 格式的工具 schema 长这样：

In [ ]:
import json

# 这是发给 LLM 的工具描述格式
example_schema = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "读取指定文件的内容。支持分页（offset/limit）以避免读取过大文件。",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "文件路径（相对于工作目录或绝对路径）"
                },
                "offset": {
                    "type": "integer",
                    "description": "从第几行开始读（从 1 开始），不填则从头读"
                },
                "limit": {
                    "type": "integer",
                    "description": "最多读多少行，不填则读全部"
                }
            },
            "required": ["path"]  # 必填参数
        }
    }
}

print(json.dumps(example_schema, indent=2, ensure_ascii=False))

**`description` 字段非常重要。** LLM 根据 description 判断什么时候用这个工具。写得越清楚，LLM 调用得越准确。

不好的描述：`"读取文件"`  
好的描述：`"读取指定文件的内容。适用于需要查看源代码、配置文件或文本文件时。二进制文件不支持。"`

## 3. ToolKind：工具分类

工具按「会不会改变系统状态」分类。这个分类后面的**审批系统**（第 09 章）会用到——写文件要确认，读文件不用。

In [ ]:
from enum import Enum

class ToolKind(Enum):
    READ    = "read"    # 只读操作：读文件、列目录、搜索文件
    WRITE   = "write"   # 写操作：创建/修改文件（需要确认）
    SHELL   = "shell"   # 执行命令（危险，需要确认）
    NETWORK = "network" # 网络请求：搜索、抓取网页
    MEMORY  = "memory"  # 持久化存储：读写记忆文件
    MCP     = "mcp"     # 通过 MCP 协议调用的外部工具

print("ToolKind 定义:")
for kind in ToolKind:
    print(f"  {kind.name:8s} → {kind.value}")

## 4. ToolResult：统一的调用结果

工具执行完，不管成功还是失败，都返回 `ToolResult`。

为什么要统一格式？
- Agent 循环不用写 `if 成功 else 失败` 分支——拿到结果直接塞进 messages
- LLM 能看到完整的错误信息，可以根据错误调整重试策略
- UI 层统一渲染，不用为每个工具写不同的展示逻辑

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class ToolResult:
    """
    工具执行结果。无论成功失败，都用这个格式返回。
    
    content    : 返回给 LLM 的文本（成功时是结果，失败时是错误信息）
    success    : 是否执行成功
    metadata   : 额外数据（不发给 LLM，供 UI 展示用）
    is_error   : 标记这是错误结果（LLM 会据此调整行为）
    """
    content: str
    success: bool
    metadata: dict[str, Any] = field(default_factory=dict)
    is_error: bool = False

    # ── 工厂方法：语义清晰的构造方式 ────────────────────────

    @classmethod
    def ok(cls, content: str, **metadata) -> "ToolResult":
        """成功结果"""
        return cls(content=content, success=True, metadata=metadata)

    @classmethod
    def error(cls, message: str, **metadata) -> "ToolResult":
        """失败结果，message 会发给 LLM，让它知道出了什么问题"""
        return cls(content=f"错误：{message}", success=False, metadata=metadata, is_error=True)

    def to_tool_message(self, tool_call_id: str) -> dict:
        """转换为 OpenAI 消息格式，用于塞回 messages 列表"""
        return {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "content": self.content,
        }


# 演示成功和失败的结果
success = ToolResult.ok(
    "def hello():\n    print('Hello, World!')",
    path="hello.py",
    line_count=2,
)
print("成功结果:")
print(f"  success  = {success.success}")
print(f"  content  = {success.content[:50]}")
print(f"  metadata = {success.metadata}")
print()

failure = ToolResult.error(
    "文件不存在: /path/to/missing.py",
    path="/path/to/missing.py",
)
print("失败结果:")
print(f"  success  = {failure.success}")
print(f"  content  = {failure.content}")
print(f"  is_error = {failure.is_error}")
print()

# 转换为消息格式
print("发回给 LLM 的格式:")
print(failure.to_tool_message("call_abc123"))

## 5. Tool：抽象基类

所有工具都继承自 `Tool`，必须实现这几个方法：

| 方法 | 作用 |
|------|------|
| `name` | 工具名称（LLM 用这个名字调用） |
| `description` | 工具描述（LLM 根据这个决定什么时候用） |
| `kind` | 工具类型（READ/WRITE/SHELL...） |
| `schema()` | 返回发给 LLM 的 JSON Schema |
| `execute(params)` | 实际执行逻辑，返回 `ToolResult` |
| `validate(params)` | 验证参数合法性，返回错误列表 |
| `is_mutating()` | 这个工具会改变系统状态吗？影响审批策略 |

In [ ]:
from abc import ABC, abstractmethod


class Tool(ABC):
    """
    工具抽象基类。
    
    继承这个类，实现所有抽象方法，就能注册一个新工具。
    不需要改 Agent 的任何代码。
    """

    @property
    @abstractmethod
    def name(self) -> str:
        """工具名称，LLM 调用时用这个名字。只能是字母、数字、下划线。"""
        ...

    @property
    @abstractmethod
    def description(self) -> str:
        """给 LLM 看的工具描述，决定 LLM 什么时候会选择调用这个工具。写清楚点。"""
        ...

    @property
    @abstractmethod
    def kind(self) -> ToolKind:
        ...

    @abstractmethod
    def schema(self) -> dict:
        """
        返回 OpenAI function calling 格式的工具描述。
        这个 dict 会直接塞进发给 LLM 的 tools 列表。
        """
        ...

    @abstractmethod
    async def execute(self, params: dict) -> ToolResult:
        """
        执行工具，返回 ToolResult。
        
        无论成功失败都要返回 ToolResult，不要直接抛异常。
        异常留给 ToolRegistry.invoke() 的外层捕获处理。
        """
        ...

    @abstractmethod
    def validate(self, params: dict) -> list[str]:
        """
        验证参数，返回错误信息列表。
        空列表 = 验证通过。
        """
        ...

    def is_mutating(self) -> bool:
        """这个工具会修改文件/系统状态吗？默认 False。WRITE/SHELL 类型应该返回 True。"""
        return self.kind in (ToolKind.WRITE, ToolKind.SHELL)

    def __repr__(self) -> str:
        return f"<Tool {self.name} ({self.kind.value})>"


print("Tool 基类定义完成")

## 6. 写一个真实的工具：EchoTool

用一个简单的例子演示怎么实现工具接口。

EchoTool 把输入的文本原样返回——功能没意义，但结构完整，看清楚每个方法怎么写。

In [ ]:
class EchoTool(Tool):
    """
    把输入的文本原样返回。用来测试工具框架是否正常工作。
    """

    @property
    def name(self) -> str:
        return "echo"

    @property
    def description(self) -> str:
        # 对真实工具来说，这里要写清楚「什么时候该用」
        return "把输入的文本原样返回。用于测试工具调用是否正常工作。"

    @property
    def kind(self) -> ToolKind:
        return ToolKind.READ  # 不修改任何状态

    def schema(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "text": {
                            "type": "string",
                            "description": "要回显的文本"
                        },
                        "repeat": {
                            "type": "integer",
                            "description": "重复几次，默认 1",
                        }
                    },
                    "required": ["text"]
                }
            }
        }

    def validate(self, params: dict) -> list[str]:
        errors = []
        if "text" not in params:
            errors.append("缺少必填参数: text")
        if "repeat" in params:
            if not isinstance(params["repeat"], int):
                errors.append("repeat 必须是整数")
            elif params["repeat"] < 1 or params["repeat"] > 10:
                errors.append("repeat 范围是 1-10")
        return errors

    async def execute(self, params: dict) -> ToolResult:
        text = params["text"]
        repeat = params.get("repeat", 1)
        result = "\n".join([text] * repeat)
        return ToolResult.ok(result, char_count=len(result), repeat=repeat)


# 快速验证
tool = EchoTool()
print(repr(tool))
print("是否修改状态:", tool.is_mutating())
print()

# 参数验证
print("验证空参数:", tool.validate({}))
print("验证正确参数:", tool.validate({"text": "hello", "repeat": 3}))
print("验证超范围参数:", tool.validate({"text": "hi", "repeat": 99}))

In [ ]:
# 执行工具
result = await tool.execute({"text": "Hello, Agent!", "repeat": 3})
print("执行结果:")
print(f"  success  = {result.success}")
print(f"  content  =\n{result.content}")
print(f"  metadata = {result.metadata}")
print()
print("转换为消息格式:")
print(result.to_tool_message("call_test_001"))

## 7. ToolRegistry：工具注册表

Agent 不直接调用工具类，而是通过注册表来调用。

注册表的职责：
1. **管理工具**：注册、注销
2. **提供 schema**：把所有工具的 JSON Schema 打包，发给 LLM
3. **执行调用**：收到 LLM 的工具调用指令，验证参数，执行，返回结果

为什么要这层抽象？
- Agent 代码不需要知道有哪些工具，只和注册表交互
- 加新工具不改 Agent 代码，只在注册表里 `register()`
- 参数验证和异常捕获集中在这里，工具本身不用重复写

In [ ]:
import logging

logger = logging.getLogger(__name__)


class ToolRegistry:
    """
    工具注册表。Agent 通过这里调用所有工具。
    """

    def __init__(self):
        self._tools: dict[str, Tool] = {}

    # ── 注册管理 ──────────────────────────────────────────────

    def register(self, tool: Tool):
        if tool.name in self._tools:
            logger.warning(f"工具 '{tool.name}' 已存在，将被覆盖")
        self._tools[tool.name] = tool

    def unregister(self, name: str) -> bool:
        if name in self._tools:
            del self._tools[name]
            return True
        return False

    def get(self, name: str) -> Tool | None:
        return self._tools.get(name)

    def list_tools(self) -> list[Tool]:
        return list(self._tools.values())

    # ── Schema 生成（发给 LLM）────────────────────────────────

    def get_schemas(self) -> list[dict]:
        """
        返回所有工具的 JSON Schema 列表，直接传给 LLM 的 tools 参数。
        """
        return [tool.schema() for tool in self._tools.values()]

    # ── 调用工具（Agent 主循环调用这个）─────────────────────────

    async def invoke(self, name: str, params: dict) -> ToolResult:
        """
        执行一次工具调用。
        
        流程：找工具 → 验证参数 → 执行 → 捕获异常 → 返回 ToolResult
        无论发生什么，都返回 ToolResult，不向外抛异常。
        """
        # 1. 找工具
        tool = self.get(name)
        if tool is None:
            return ToolResult.error(
                f"未知工具: '{name}'。可用工具: {list(self._tools.keys())}"
            )

        # 2. 验证参数
        errors = tool.validate(params)
        if errors:
            error_msg = "参数验证失败:\n" + "\n".join(f"  - {e}" for e in errors)
            return ToolResult.error(error_msg, tool=name, params=params)

        # 3. 执行（捕获所有异常，包装成 error ToolResult）
        try:
            result = await tool.execute(params)
            return result
        except Exception as e:
            logger.exception(f"工具 '{name}' 执行时发生未预期异常")
            return ToolResult.error(
                f"工具执行时发生内部错误: {type(e).__name__}: {e}",
                tool=name,
            )

    def __len__(self):
        return len(self._tools)

    def __repr__(self):
        names = list(self._tools.keys())
        return f"<ToolRegistry [{', '.join(names)}]>"


print("ToolRegistry 定义完成")

## 8. 注册和调用工具

In [ ]:
registry = ToolRegistry()
registry.register(EchoTool())

print(registry)
print(f"注册了 {len(registry)} 个工具")
print()

# 看看发给 LLM 的 schema 长什么样
schemas = registry.get_schemas()
print("发给 LLM 的 tool schemas:")
print(json.dumps(schemas, indent=2, ensure_ascii=False))

In [ ]:
# 正常调用
result = await registry.invoke("echo", {"text": "工具框架测试", "repeat": 2})
print("正常调用:")
print(f"  success = {result.success}")
print(f"  content = {result.content}")
print()

In [ ]:
# 参数错误
result = await registry.invoke("echo", {"repeat": 3})  # 缺少必填的 text
print("参数错误:")
print(f"  success  = {result.success}")
print(f"  is_error = {result.is_error}")
print(f"  content  = {result.content}")
print()

In [ ]:
# 工具不存在
result = await registry.invoke("read_file", {"path": "main.py"})
print("工具不存在:")
print(f"  success  = {result.success}")
print(f"  content  = {result.content}")
print()

In [ ]:
# 测试异常捕获：一个执行时会崩的工具
class BrokenTool(Tool):
    @property
    def name(self): return "broken"
    @property
    def description(self): return "一个会崩的工具"
    @property
    def kind(self): return ToolKind.READ
    def schema(self): return {"type": "function", "function": {"name": "broken", "description": "", "parameters": {"type": "object", "properties": {}}}}
    def validate(self, params): return []
    async def execute(self, params):
        raise RuntimeError("Something went terribly wrong")

registry.register(BrokenTool())
result = await registry.invoke("broken", {})
print("异常工具被捕获:")
print(f"  success  = {result.success}")
print(f"  is_error = {result.is_error}")
print(f"  content  = {result.content}")
# 注意：没有崩溃！异常被 registry.invoke 捕获并包装成 ToolResult.error

## 9. 完整流程预演：LLM 调用工具

用一个完整的例子，把前三章的内容串起来看一遍。

LLM 收到工具列表，决定调用 echo，我们接收工具调用指令，执行，把结果发回去。

In [ ]:
import json
import sys
sys.path.insert(0, "..")

from src.llm_client import LLMClient
from src.context_manager import ContextManager

# 初始化
llm = LLMClient()
ctx = ContextManager(system_prompt="你是一个助手。有需要时使用工具。")
reg = ToolRegistry()
reg.register(EchoTool())

print("初始化完成")
print(f"注册工具: {[t.name for t in reg.list_tools()]}")

In [ ]:
# 第一步：把用户消息和工具列表发给 LLM
ctx.add_user_message("请用 echo 工具把 'Hello from tool!' 这段文字重复 3 次")

# 发请求（注意：这里直接调 LLM 客户端，第 06 章会封装成 Agent 循环）
from openai import AsyncOpenAI
raw_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

response = await raw_client.chat.completions.create(
    model="gpt-oss:120b",
    messages=ctx.get_messages(),
    tools=reg.get_schemas(),  # 把工具描述发给 LLM
)

msg = response.choices[0].message
print("LLM 回复:")
print(f"  finish_reason : {response.choices[0].finish_reason}")
print(f"  content       : {msg.content}")
print(f"  tool_calls    : {msg.tool_calls}")

In [ ]:
# 第二步：如果 LLM 想调工具，执行它
if msg.tool_calls:
    # 把 LLM 的工具调用意图加进 messages（必须保留这条，否则 LLM 会迷失）
    ctx.add_assistant_message(
        content=msg.content or "",
        tool_calls=[tc.model_dump() for tc in msg.tool_calls],
    )

    # 执行每个工具调用
    for tool_call in msg.tool_calls:
        name = tool_call.function.name
        params = json.loads(tool_call.function.arguments)
        call_id = tool_call.id

        print(f"\n执行工具: {name}")
        print(f"  参数: {params}")

        # 通过注册表执行
        result = await reg.invoke(name, params)
        print(f"  结果: {result.content[:100]}")

        # 把工具结果加回 messages
        ctx.add_tool_result(call_id, result.content)

    # 第三步：把工具结果发给 LLM，让它继续回答
    response2 = await raw_client.chat.completions.create(
        model="gpt-oss:120b",
        messages=ctx.get_messages(),
        tools=reg.get_schemas(),
    )

    print("\nLLM 最终回复:")
    print(response2.choices[0].message.content)
else:
    print("LLM 没有调用工具，直接回答:")
    print(msg.content)

await raw_client.close()

上面这段「判断是否有 tool_calls → 执行 → 把结果发回去」的逻辑，就是 **Agent 循环的核心**。

第 06 章会把这个循环封装成 `AgenticLoop`，加上循环处理（LLM 可能连续调用多个工具）、错误重试、事件流等。

## 10. 小结

| 组件 | 作用 |
|------|------|
| `ToolKind` | 按「是否改变状态」分类工具，影响审批策略 |
| `ToolResult` | 统一的执行结果：`.ok()` / `.error()`，LLM 和 UI 都用这个 |
| `Tool` | 抽象基类，规定每个工具必须实现的接口 |
| `ToolRegistry` | 注册表：集中管理工具，统一做参数验证和异常捕获 |
| `get_schemas()` | 把所有工具转成 LLM 能理解的 JSON Schema，传给 API |
| `invoke()` | Agent 调用工具的唯一入口，包含完整的错误处理 |

**这个框架的设计原则**：工具本身只关心「怎么做事」，注册表负责「安不安全、参数对不对、出错了怎么办」。两层职责分离，加新工具时不用改任何基础设施。

**下一章：File Tools**  
有了框架，开始造真正有用的工具——读文件、写文件、搜索文件。这三个工具是 Coding Agent 最高频使用的操作，细节很多（路径安全、大文件处理、二进制检测……），单独一章讲清楚。

---
## 附：保存到 src/

In [ ]:
tool_framework_code = '''
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from typing import Any

logger = logging.getLogger(__name__)


class ToolKind(Enum):
    READ    = "read"
    WRITE   = "write"
    SHELL   = "shell"
    NETWORK = "network"
    MEMORY  = "memory"
    MCP     = "mcp"


@dataclass
class ToolResult:
    content: str
    success: bool
    metadata: dict[str, Any] = field(default_factory=dict)
    is_error: bool = False

    @classmethod
    def ok(cls, content: str, **metadata) -> "ToolResult":
        return cls(content=content, success=True, metadata=metadata)

    @classmethod
    def error(cls, message: str, **metadata) -> "ToolResult":
        return cls(content=f"错误：{message}", success=False, metadata=metadata, is_error=True)

    def to_tool_message(self, tool_call_id: str) -> dict:
        return {"role": "tool", "tool_call_id": tool_call_id, "content": self.content}


class Tool(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...

    @property
    @abstractmethod
    def description(self) -> str: ...

    @property
    @abstractmethod
    def kind(self) -> ToolKind: ...

    @abstractmethod
    def schema(self) -> dict: ...

    @abstractmethod
    async def execute(self, params: dict) -> ToolResult: ...

    @abstractmethod
    def validate(self, params: dict) -> list[str]: ...

    def is_mutating(self) -> bool:
        return self.kind in (ToolKind.WRITE, ToolKind.SHELL)

    def __repr__(self):
        return f"<Tool {self.name} ({self.kind.value})>"


class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, Tool] = {}

    def register(self, tool: Tool):
        if tool.name in self._tools:
            logger.warning(f"工具 \'{tool.name}\' 已存在，将被覆盖")
        self._tools[tool.name] = tool

    def unregister(self, name: str) -> bool:
        if name in self._tools:
            del self._tools[name]
            return True
        return False

    def get(self, name: str) -> Tool | None:
        return self._tools.get(name)

    def list_tools(self) -> list[Tool]:
        return list(self._tools.values())

    def get_schemas(self) -> list[dict]:
        return [tool.schema() for tool in self._tools.values()]

    async def invoke(self, name: str, params: dict) -> ToolResult:
        tool = self.get(name)
        if tool is None:
            return ToolResult.error(f"未知工具: \'{name}\'。可用工具: {list(self._tools.keys())}")
        errors = tool.validate(params)
        if errors:
            return ToolResult.error("参数验证失败:\\n" + "\\n".join(f"  - {e}" for e in errors), tool=name)
        try:
            return await tool.execute(params)
        except Exception as e:
            logger.exception(f"工具 \'{name}\' 执行异常")
            return ToolResult.error(f"工具执行内部错误: {type(e).__name__}: {e}", tool=name)

    def __len__(self): return len(self._tools)
    def __repr__(self): return f"<ToolRegistry [{\', \'.join(self._tools.keys())}]>"
'''

with open("src/tool_framework.py", "w") as f:
    f.write(tool_framework_code.strip())

print("已保存到 src/tool_framework.py")
print("后续章节用法: from src.tool_framework import Tool, ToolKind, ToolResult, ToolRegistry")

In [ ]:
await llm.close()